# Génération de la couche Bronze

Ce notebook transforme les trois sources du dossier `datas` en fichiers CSV UTF-8 dans `artifacts/ingestion/bronze`. Les deux fichiers CSV produisent chacun une sortie. Le dump SQL produit une sortie par table, soit quatre fichiers Bronze au total.

## Règles appliquées

- les en-têtes et l’ordre des colonnes d’origine sont conservés ;
- chaque valeur est copiée sous forme de texte, sans changement de casse, d’espaces ou de contenu ;
- chaque colonne d’origine est immédiatement suivie de `<colonne>_type` ;
- la dernière colonne, `row_sha256`, contient le SHA-256 salé des seules valeurs d’origine ;
- les noms de sortie sont suffixés par `AAAAMMJJ_HHMMSS` ;
- le traitement s’arrête à la première erreur avec un diagnostic explicite.

## 1. Imports et paramètres

pandas lit et écrit les fichiers plats. SQLAlchemy charge le dump dans une base SQLite en mémoire et expose ses tables. Le sel n’est jamais stocké dans le notebook : il provient de la variable utilisateur Windows `INDUSENSE_HASH_SALT`.

In [1]:
from __future__ import annotations

import csv
import hashlib
import os
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from sqlalchemy import MetaData, String, cast, create_engine, inspect, select
from sqlalchemy.sql.sqltypes import Boolean, Date, DateTime, Float, Integer, Numeric

INPUT_DIRECTORY = Path("datas")
OUTPUT_DIRECTORY = Path("artifacts/ingestion/bronze")
HASH_COLUMN = "row_sha256"
HASH_SEPARATOR = "\x1f"
SALT_SEPARATOR = "\x1e"


@dataclass(frozen=True)
class BronzeResult:
    source: Path
    dataset: str
    output: Path
    rows: int
    source_columns: int
    output_columns: int

## 2. Fonctions communes

La transformation travaille uniquement avec des chaînes. Pour les CSV, les valeurs sont lues directement comme texte et les valeurs vides ne sont pas converties en `NaN`. Pour SQL, chaque valeur sélectionnée est convertie en texte par le moteur.

Le hash est calculé sur les valeurs d’origine dans l’ordre des colonnes. Aucune normalisation de contenu n’est appliquée. Le caractère de contrôle `\x1F` sert uniquement de séparateur entre les valeurs afin de réduire les ambiguïtés de concaténation. Le sel est préfixé et séparé par `\x1E`.

In [2]:
def load_hash_salt() -> str:
    salt = os.environ.get("INDUSENSE_HASH_SALT")
    if not salt and os.name == "nt":
        import winreg
        try:
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as key:
                salt, _ = winreg.QueryValueEx(key, "INDUSENSE_HASH_SALT")
        except FileNotFoundError:
            salt = None
    if not salt:
        raise EnvironmentError(
            "La variable INDUSENSE_HASH_SALT est absente ou vide."
        )
    return str(salt)


def detect_charset(path: Path) -> tuple[str, str]:
    content = path.read_bytes()
    if content.startswith(b"\xef\xbb\xbf"):
        return "UTF-8 BOM", "utf-8-sig"
    for label, codec in (
        ("UTF-8", "utf-8"),
        ("Windows-1252", "cp1252"),
        ("ISO-8859-1", "iso-8859-1"),
    ):
        try:
            content.decode(codec)
            return label, codec
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Impossible de déterminer le charset de {path}")


def detect_delimiter(path: Path, encoding: str) -> str:
    with path.open("r", encoding=encoding, newline="") as stream:
        sample = stream.read(16_384)
    if not sample:
        raise ValueError("Le fichier CSV est vide.")
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error as exc:
        raise ValueError("Séparateur CSV indéterminable.") from exc


def infer_text_type(value: str, *, is_null: bool = False) -> str:
    if is_null:
        return "NULL"
    if value == "":
        return "EMPTY"
    if re.fullmatch(r"(?i:true|false)", value):
        return "BOOLEAN"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:\.\d+)?(?:Z|[+-]\d{2}(?::?\d{2})?)?", value):
        return "DATETIME"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return "DATE"
    if re.fullmatch(r"\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?", value):
        return "TIME"
    if re.fullmatch(r"[+-]?\d+", value):
        return "INTEGER"
    if re.fullmatch(r"[+-]?(?:\d+\.\d*|\d*\.\d+)(?:[eE][+-]?\d+)?|[+-]?\d+[eE][+-]?\d+", value):
        return "FLOAT"
    return "TEXT"


def compute_row_hash(values: list[str], salt: str) -> str:
    payload = salt + SALT_SEPARATOR + HASH_SEPARATOR.join(values)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def build_bronze_frame(
    source_frame: pd.DataFrame,
    salt: str,
    explicit_types: dict[str, str] | None = None,
    null_masks: dict[str, pd.Series] | None = None,
) -> pd.DataFrame:
    original_columns = [str(column) for column in source_frame.columns]
    text_frame = source_frame.copy()
    text_frame.columns = original_columns
    text_frame = text_frame.astype("string").fillna("")

    output = pd.DataFrame(index=text_frame.index)
    for column in original_columns:
        values = text_frame[column].astype(str)
        output[column] = values
        if explicit_types and column in explicit_types:
            inferred = pd.Series(explicit_types[column], index=text_frame.index, dtype="string")
            if null_masks and column in null_masks:
                inferred = inferred.mask(null_masks[column], "NULL")
            output[f"{column}_type"] = inferred
        else:
            output[f"{column}_type"] = values.map(infer_text_type)

    output[HASH_COLUMN] = [
        compute_row_hash([str(value) for value in row], salt)
        for row in text_frame[original_columns].itertuples(index=False, name=None)
    ]
    return output


def write_bronze_csv(frame: pd.DataFrame, output: Path) -> None:
    output.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        output,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        quoting=csv.QUOTE_ALL,
    )

## 3. Traitement des fichiers plats

`dtype=str`, `keep_default_na=False` et `na_filter=False` empêchent pandas de convertir les valeurs. Le fichier Bronze conserve ainsi le texte logique de chaque champ CSV. Toutes les cellules sont entourées de guillemets lors de l’écriture pour expliciter leur nature textuelle.

In [3]:
def process_flat_file(
    source: Path, output_directory: Path, timestamp: str, salt: str
) -> BronzeResult:
    _, encoding = detect_charset(source)
    delimiter = detect_delimiter(source, encoding)
    source_frame = pd.read_csv(
        source,
        encoding=encoding,
        sep=delimiter,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
    )
    bronze_frame = build_bronze_frame(source_frame, salt)
    output = output_directory / f"{source.stem}_{timestamp}.csv"
    write_bronze_csv(bronze_frame, output)
    return BronzeResult(
        source=source.resolve(),
        dataset=source.stem,
        output=output.resolve(),
        rows=len(source_frame),
        source_columns=len(source_frame.columns),
        output_columns=len(bronze_frame.columns),
    )

## 4. Traitement du dump SQL

Le dump PostgreSQL fourni contient deux tables. Deux adaptations syntaxiques ciblées permettent son chargement dans SQLite en mémoire, sans modifier la source. SQLAlchemy réfléchit ensuite les tables et sélectionne chaque valeur avec un `CAST` texte. Les types associés proviennent du schéma SQL réfléchi ; une valeur SQL `NULL` reçoit toujours le type `NULL`.

In [4]:
def sqlite_compatible_sql(sql: str) -> str:
    sql = re.sub(r"\bTIMESTAMPTZ\b", "TIMESTAMP", sql, flags=re.IGNORECASE)
    return re.sub(r"\bNOW\s*\(\s*\)", "CURRENT_TIMESTAMP", sql, flags=re.IGNORECASE)


def declared_sql_type(column: Any) -> str:
    if isinstance(column.type, Boolean):
        return "BOOLEAN"
    if isinstance(column.type, DateTime):
        return "DATETIME"
    if isinstance(column.type, Date):
        return "DATE"
    if isinstance(column.type, Integer):
        return "INTEGER"
    if isinstance(column.type, (Numeric, Float)):
        return "FLOAT"
    return "TEXT"


def load_sql_tables(source: Path) -> tuple[Any, MetaData]:
    _, encoding = detect_charset(source)
    sql = source.read_text(encoding=encoding)
    if not sql.strip():
        raise ValueError("Le fichier SQL est vide.")
    engine = create_engine("sqlite+pysqlite:///:memory:")
    try:
        raw_connection = engine.raw_connection()
        try:
            raw_connection.executescript(sqlite_compatible_sql(sql))
            raw_connection.commit()
        finally:
            raw_connection.close()
        metadata = MetaData()
        metadata.reflect(bind=engine)
        if not metadata.tables:
            raise ValueError("Aucune table trouvée dans le fichier SQL.")
        return engine, metadata
    except Exception:
        engine.dispose()
        raise


def process_sql_file(
    source: Path, output_directory: Path, timestamp: str, salt: str
) -> list[BronzeResult]:
    engine, metadata = load_sql_tables(source)
    results: list[BronzeResult] = []
    try:
        with engine.connect() as connection:
            for table_name in inspect(engine).get_table_names():
                table = metadata.tables[table_name]
                columns = list(table.columns)
                statement = select(*[cast(column, String).label(column.name) for column in columns])
                rows = connection.execute(statement).mappings().all()
                source_frame = pd.DataFrame(
                    [[row[column.name] for column in columns] for row in rows],
                    columns=[column.name for column in columns],
                )
                null_masks = {
                    column.name: source_frame[column.name].isna() for column in columns
                }
                explicit_types = {
                    column.name: declared_sql_type(column) for column in columns
                }
                bronze_frame = build_bronze_frame(
                    source_frame, salt, explicit_types=explicit_types, null_masks=null_masks
                )
                output = output_directory / f"{table_name}_{timestamp}.csv"
                write_bronze_csv(bronze_frame, output)
                persisted = pd.read_csv(
                    output, dtype=str, keep_default_na=False, na_filter=False
                )
                pd.testing.assert_frame_equal(
                    source_frame.astype("string").fillna("").reset_index(drop=True),
                    persisted[[column.name for column in columns]].reset_index(drop=True),
                    check_dtype=False,
                )
                results.append(BronzeResult(
                    source=source.resolve(),
                    dataset=table_name,
                    output=output.resolve(),
                    rows=len(source_frame),
                    source_columns=len(columns),
                    output_columns=len(bronze_frame.columns),
                ))
    finally:
        engine.dispose()
    return results

## 5. Orchestration et arrêt sur erreur

Un horodatage unique est créé au début du lot afin que les quatre fichiers soient identifiables comme une même génération. Le traitement reste séquentiel et s’interrompt dès qu’une source échoue.

In [5]:
def generate_bronze_files(
    input_directory: Path = INPUT_DIRECTORY,
    output_directory: Path = OUTPUT_DIRECTORY,
) -> list[BronzeResult]:
    salt = load_hash_salt()
    timestamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
    flat_sources = sorted(input_directory.glob("*.csv"))
    sql_sources = sorted(input_directory.glob("*.sql.txt"))
    if not flat_sources and not sql_sources:
        raise FileNotFoundError(f"Aucune source trouvée dans {input_directory.resolve()}")

    results: list[BronzeResult] = []
    for source in [*flat_sources, *sql_sources]:
        try:
            if source.suffix.casefold() == ".csv":
                results.append(process_flat_file(source, output_directory, timestamp, salt))
            else:
                results.extend(process_sql_file(source, output_directory, timestamp, salt))
        except Exception as exc:
            print(f"ERREUR pendant le traitement de {source.resolve()}")
            print(f"{type(exc).__name__}: {exc}")
            raise RuntimeError(f"Traitement Bronze interrompu sur {source.name}") from exc

    print(f"Traitement Bronze terminé : {len(results)} fichier(s) généré(s).")
    for result in results:
        print(
            f"- {result.dataset}: {result.rows} ligne(s), "
            f"{result.source_columns} colonne(s) source -> {result.output}"
        )
    return results

## 6. Génération des fichiers Bronze

In [6]:
results = generate_bronze_files()

summary = pd.DataFrame([
    {
        "Source": item.source.name,
        "Jeu de données": item.dataset,
        "Lignes": item.rows,
        "Colonnes source": item.source_columns,
        "Colonnes Bronze": item.output_columns,
        "Fichier Bronze": str(item.output),
    }
    for item in results
])
summary

Traitement Bronze terminé : 4 fichier(s) généré(s).
- releves_incidents: 1245 ligne(s), 18 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\releves_incidents_20261126_170306.csv
- telemetry: 135626 ligne(s), 7 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\telemetry_20261126_170306.csv
- machine: 15 ligne(s), 11 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\machine_20261126_170306.csv
- maintenance: 115 ligne(s), 11 colonne(s) source -> D:\indusense\artifacts\ingestion\bronze\maintenance_20261126_170306.csv


,Source,Jeu de données,Lignes,Colonnes source,Colonnes Bronze,Fichier Bronze
0,releves_incidents.csv,releves_incidents,1245,18,37,D:\indusense\artifacts\ingestion\bronze\releve...
1,telemetry.csv,telemetry,135626,7,15,D:\indusense\artifacts\ingestion\bronze\teleme...
2,machine.sql.txt,machine,15,11,23,D:\indusense\artifacts\ingestion\bronze\machin...
3,machine.sql.txt,maintenance,115,11,23,D:\indusense\artifacts\ingestion\bronze\mainte...


## 7. Contrôles

Les contrôles vérifient le nombre de sorties, les dimensions, l’alternance des colonnes, l’encodage UTF-8, la copie exacte des deux CSV et des deux tables SQL, ainsi que le recalcul des empreintes. Le sel reste secret et n’est jamais affiché.

In [7]:
expected = {
    "releves_incidents": (1245, 18),
    "telemetry": (135626, 7),
    "machine": (15, 11),
    "maintenance": (115, 11),
}
assert len(results) == 4
salt = load_hash_salt()

for result in results:
    expected_rows, expected_source_columns = expected[result.dataset]
    assert result.rows == expected_rows
    assert result.source_columns == expected_source_columns
    assert result.output_columns == expected_source_columns * 2 + 1
    result.output.read_text(encoding="utf-8")

    bronze = pd.read_csv(
        result.output, dtype=str, keep_default_na=False, na_filter=False
    )
    original_columns = list(bronze.columns[0:-1:2])
    expected_headers = [
        header
        for column in original_columns
        for header in (column, f"{column}_type")
    ] + [HASH_COLUMN]
    assert list(bronze.columns) == expected_headers
    assert bronze[HASH_COLUMN].str.fullmatch(r"[0-9a-f]{64}").all()
    recomputed = [
        compute_row_hash([str(value) for value in row], salt)
        for row in bronze[original_columns].itertuples(index=False, name=None)
    ]
    assert recomputed == bronze[HASH_COLUMN].tolist()

for result in (item for item in results if item.source.suffix.casefold() == ".csv"):
    _, encoding = detect_charset(result.source)
    delimiter = detect_delimiter(result.source, encoding)
    source_frame = pd.read_csv(
        result.source, encoding=encoding, sep=delimiter, dtype=str,
        keep_default_na=False, na_filter=False,
    )
    bronze = pd.read_csv(
        result.output, dtype=str, keep_default_na=False, na_filter=False
    )
    pd.testing.assert_frame_equal(
        source_frame.reset_index(drop=True),
        bronze[list(source_frame.columns)].reset_index(drop=True),
        check_dtype=False,
    )

print("Contrôles réussis : 4 sorties UTF-8, valeurs CSV/SQL intactes, colonnes et hash validés.")

Contrôles réussis : 4 sorties UTF-8, valeurs CSV/SQL intactes, colonnes et hash validés.


## Synthèse des choix

- **CSV UTF-8 avec guillemets systématiques** : les sorties restent des fichiers plats et toutes les valeurs sont explicitement textuelles.
- **Une sortie par source logique** : deux fichiers plats et deux tables SQL produisent quatre fichiers.
- **Colonnes intercalées** : chaque valeur est immédiatement suivie de son type, ce qui facilite les contrôles ligne par ligne.
- **Aucune normalisation des valeurs** : le hash reprend le texte sans `strip`, changement de casse ou remplacement.
- **Sel externe persistant** : la variable utilisateur Windows protège le sel et stabilise les empreintes entre les exécutions.
- **Horodatage commun au lot** : les quatre noms de fichiers portent la date et l’heure d’une même génération.